In [1]:
import xarray as xr
import matplotlib.pyplot as plt
import numpy as np
import warnings

import cartopy.crs as ccrs
import cartopy.mpl.ticker as cticker
import cartopy.feature as cfeature


import proplot as pplt

from sklearn.cluster import KMeans

from eofs.multivariate.standard import MultivariateEof
from eofs.xarray import Eof

warnings.filterwarnings("ignore")

In [2]:
def daily_climo(da,varname,**kwargs):
  
    # This function is adapted the code written by Ray Bell for the SubX project; it is for the
    # verification data
    
    clim_fname = kwargs.get('fname', None)
    
    # Average daily data
    da_day_clim = da.groupby('time.dayofyear').mean('time')
    
    # Rechunk for time
    da_day_clim = da_day_clim.chunk({'dayofyear': 366})
    
    # Pad the daily climatolgy with nans
    x = np.empty((366, len(da_day_clim.lat), len(da_day_clim.lon)))
    x.fill(np.nan)
    _da = xr.DataArray(x,name=varname, coords=[np.linspace(1, 366, num=366, dtype=np.int64),
                              da_day_clim.lat, da_day_clim.lon],
                              dims = da_day_clim.dims)
    da_day_clim_wnan = da_day_clim.combine_first(_da)

    
    # Period rolling twice to make it triangular smoothing
    # See https://bit.ly/2H3o0Mf
    da_day_clim_smooth = da_day_clim_wnan.copy()
 
    

    for i in range(2):
        # Extand the DataArray to allow rolling to do periodic
        da_day_clim_smooth = xr.concat([da_day_clim_smooth[-15:],
                                        da_day_clim_smooth,
                                        da_day_clim_smooth[:15]],
                                        'dayofyear')
        # Rolling mean
        da_day_clim_smooth = da_day_clim_smooth.rolling(dayofyear=31,
                                                        center=True,
                                                        min_periods=1).mean()
        # Drop the periodic boundaries
        da_day_clim_smooth = da_day_clim_smooth.isel(dayofyear=slice(15, -15))

    
    # Extract the original days
    da_day_clim_smooth = da_day_clim_smooth.sel(dayofyear=da_day_clim.dayofyear)

    da_day_clim_smooth.name=varname
    ds_day_clim_smooth=da_day_clim_smooth.to_dataset()
    
    # Save to file if filename provide and return True, otherwise return the data


In [3]:
from dask.distributed import Client
from dask.distributed import LocalCluster
cluster = LocalCluster()
cluster

LocalCluster(9285de1f, 'tcp://127.0.0.1:36723', workers=13, threads=104, memory=1.97 TiB)

In [4]:
# Region
min_lat = 10
max_lat = 70
#min_lon = 150 #150 E
min_lon = 360-150 #150W
max_lon = 360-40 #40 W

# Date
#sdate = '1979-01-01'
#edate = '2019-12-31'
#sdate = '1999-01-01'
#edate = '2019-12-31'
sdate = '1981-01-01'
edate = '2019-12-31'


# Month
seas='DJF'
seas_mon=[1,2,12]
#seas='SONDJFM'
#seas_mon=[9,10,11,12,1,2,3]

npcs = 12

data_path = '/data/esplab/scratch/kpegion/obs_seus/wr_z500/'

In [5]:
inpath = '/data/esplab/shared/reanalysis/era5/daily/z500/'
ifname = 'z.*.nc'
ds_z = xr.open_mfdataset(inpath+ifname,combine='nested',concat_dim='time')
ds_z = ds_z.rename({'latitude':'lat','longitude':'lon'})
ds_z = ds_z.reindex(lat=list(reversed(ds_z['lat'])))
ds_z['z']=ds_z['z']/9.81

In [6]:
ds_z

<xarray.Dataset> Size: 8GB
Dimensions:  (time: 30316, lon: 360, lat: 181)
Coordinates:
  * time     (time) datetime64[ns] 243kB 1940-01-01 1940-01-02 ... 2022-12-31
  * lon      (lon) float32 1kB 0.0 1.0 2.0 3.0 4.0 ... 356.0 357.0 358.0 359.0
  * lat      (lat) float32 724B -90.0 -89.0 -88.0 -87.0 ... 87.0 88.0 89.0 90.0
    level    int32 4B 500
Data variables:
    z        (time, lat, lon) float32 8GB dask.array<chunksize=(366, 181, 360), meta=np.ndarray>
Attributes:
    comments:      Daily data created from: mean of 4xdaily
    source:        Downloaded from Copernicus Data Store: https://cds.climate...
    CreationDate:  2023-07-19
    CreatedBy:     kpegion
    Source:        makeDaily.ipynb

In [7]:
# Make Anoms
ds_climo=daily_climo(ds_z['z'],'z')
ds_anoms=ds_z['z'].groupby('time.dayofyear')-ds_climo

# 5-day running means
ds_zanom=ds_anoms.rolling(time=5,center=True).mean().dropna('time')

# Subset months
ds_zanom = ds_zanom.sel(time=ds_zanom['time.month'].isin(seas_mon))

# Select the dates
ds_zanom=ds_zanom.sel(time=slice(sdate,edate))

TypeError: GroupBy objects only support binary ops when the other argument is a Dataset or DataArray